# SoulX-FlashHead: Talking Head Generator

Go to **Settings** > **Accelerator** > **T4 GPU** before running. Ensure **Internet** is switched **ON** in the right sidebar.

In [1]:
# @title 1. Setup Environment
import os
import sys
import shutil
import re

# 1. Cleanup & Clone fresh in /kaggle/working
kb_dir = '/kaggle/working/SoulX-FlashHead'
if os.path.exists(kb_dir):
    if os.path.exists(os.path.join(kb_dir, 'SoulX-FlashHead')):
        print("🧹 Cleaning up nested directories...")
        shutil.rmtree(kb_dir)

if not os.path.exists(kb_dir):
    print("📥 Cloning SoulX-FlashHead...")
    !git clone https://github.com/QasimSalemm/SoulX-FlashHead.git {kb_dir}

%cd {kb_dir}

!apt-get install -y ffmpeg -q

# 2. Fix Dependencies (Aggressive Pinning)
print("🔧 Installing dependencies...")
!pip install -q --upgrade "transformers==4.44.2" "sentencepiece==0.1.99" "protobuf<5.0.0" 
!pip install -q xformers
!pip install -q "gradio>=5.0.0" diffusers accelerate tqdm imageio easydict ftfy imageio-ffmpeg scikit-image loguru pyloudnorm decord librosa flask huggingface_hub
!pip install -q "xfuser>=0.4.3" yunchang distvae beautifulsoup4 einops

# 3. Transformers MT5 Patch
import transformers
try:
    from transformers import MT5Tokenizer
    print("✅ MT5Tokenizer correctly imported!")
except (ImportError, AttributeError):
    print("⚠️ MT5Tokenizer missing, applying monkey-patch...")
    from transformers import T5Tokenizer
    transformers.MT5Tokenizer = T5Tokenizer
    sys.modules['transformers'].MT5Tokenizer = T5Tokenizer

# 5. Face Handler (Fixed paths)
os.makedirs("flash_head/utils", exist_ok=True)
with open("flash_head/utils/cpu_face_handler.py", "w") as f:
    f.write('''import cv2\nimport numpy as np\nimport os\nfrom typing import Tuple, List\n\nclass CPUFaceHandler:\n    def __init__(self, model_selection: int = 1, min_detection_confidence: float = 0.0):\n        proto = "/kaggle/working/SoulX-FlashHead/deploy.prototxt"\n        caffe = "/kaggle/working/SoulX-FlashHead/res10_300x300_ssd_iter_140000.caffemodel"\n        self.use_dnn = False\n        import urllib.request\n        if not os.path.exists(proto):\n            try: urllib.request.urlretrieve("https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt", proto)\n            except: pass\n        if not os.path.exists(caffe):\n            try: urllib.request.urlretrieve("https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel", caffe)\n            except: pass\n        if os.path.exists(proto) and os.path.exists(caffe):\n            self.net = cv2.dnn.readNetFromCaffe(proto, caffe)\n            self.use_dnn = True\n        else:\n            cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"\n            self.cascade = cv2.CascadeClassifier(cascade_path)\n\n    def detect(self, image: np.ndarray) -> Tuple[List, List]:\n        bboxs, scores = [], []\n        img_h, img_w = image.shape[:2]\n        if self.use_dnn:\n            blob = cv2.dnn.blobFromImage(image, 1.0, (300, 300), (104.0, 177.0, 123.0))\n            self.net.setInput(blob)\n            detections = self.net.forward()\n            for i in range(detections.shape[2]):\n                confidence = float(detections[0, 0, i, 2])\n                if confidence > 0.5:\n                    bboxs.append([float(detections[0, 0, i, 3]), float(detections[0, 0, i, 4]), float(detections[0, 0, i, 5]), float(detections[0, 0, i, 6])])\n                    scores.append(confidence)\n        else:\n            gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)\n            faces = self.cascade.detectMultiScale(gray, 1.1, 5, minSize=(30, 30))\n            for (x, y, w, h) in faces:\n                bboxs.append([x/img_w, y/img_h, (x+w)/img_w, (y+h)/img_h]); scores.append(1.0)\n        return bboxs, scores\n\n    def __call__(self, image: np.ndarray) -> Tuple[List, List]:\n        return self.detect(image)\n''')

print("✅ Setup complete")

📥 Cloning SoulX-FlashHead...
Cloning into '/kaggle/working/SoulX-FlashHead'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 83 (delta 16), reused 77 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (83/83), 5.06 MiB | 20.33 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/kaggle/working/SoulX-FlashHead
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.
🔧 Installing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 36.9 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 111.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# @title 2. Download Models
from huggingface_hub import snapshot_download
import os

os.makedirs("/kaggle/working/models", exist_ok=True)

print("📥 Downloading SoulX-FlashHead (1.3B)... may take a few mins")
snapshot_download(
    repo_id="Soul-AILab/SoulX-FlashHead-1_3B",
    local_dir="/kaggle/working/models/SoulX-FlashHead-1_3B",
    local_dir_use_symlinks=False
)

print("📥 Downloading wav2vec2-base-960h...")
snapshot_download(
    repo_id="facebook/wav2vec2-base-960h",
    local_dir="/kaggle/working/models/wav2vec2-base-960h",
    local_dir_use_symlinks=False
)

print("✅ Models downloaded successfully!")

📥 Downloading SoulX-FlashHead (1.3B)... may take a few mins


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/353 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

tf_model.h5:   0%|          | 0.00/378M [00:00<?, ?B/s]

✅ Models downloaded successfully!


In [ ]:
# @title 3. Start Launch App (Clean)
import os

with open('gradio_app.py', 'r') as f:
    code = f.read()

# Fix model paths
code = code.replace(
    'value="models/SoulX-FlashHead-1_3B"',
    'value="/kaggle/working/models/SoulX-FlashHead-1_3B"'
)
code = code.replace(
    'value="models/wav2vec2-base-960h"',
    'value="/kaggle/working/models/wav2vec2-base-960h"'
)

# Enable public link + debug
code = code.replace(
    'app.launch()',
    'app.launch(share=True, debug=True)'
)

# Keep MT5 fix (safe + needed sometimes)
patch = """
import sys, transformers
from transformers import T5Tokenizer
transformers.MT5Tokenizer = T5Tokenizer
sys.modules['transformers'].MT5Tokenizer = T5Tokenizer
"""

code = patch + code

# Save clean app
with open('gradio_app_clean.py', 'w') as f:
    f.write(code)

print("🚀 Launching Clean App...")
!python gradio_app_clean.py

🚀 Launching Clean App...
2026-03-30 12:58:12.068860: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774875492.289759    1206 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774875492.353733    1206 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774875492.873517    1206 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774875492.873553    1206 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774875492.873557    1206 computation_placer.cc:17